# 01 · Datensätze

MedQA, MedMCQA und MMLU laden und als `MCQSample` vereinheitlichen.


In [1]:
# --- Bootstrap: Projekt-Wurzel finden und importierbar machen ---
import sys, pathlib

PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])


PROJECT_ROOT: <Projektordner>/analysis
Python: 3.12.13


## Konfiguration


In [2]:
from src.utils.config import named_config

for alias in ("medqa", "medmcqa", "mmlu"):
    print(alias, "->", named_config("datasets", alias))


medqa -> {'alias': 'medqa', 'repo': 'openlifescienceai/MedQA-USMLE-4-options-hf', 'config': None, 'revision': 'main', 'default_split': 'test', 'adapter': 'medqa'}
medmcqa -> {'alias': 'medmcqa', 'repo': 'openlifescienceai/medmcqa', 'config': None, 'revision': 'main', 'default_split': 'validation', 'adapter': 'medmcqa'}
mmlu -> {'alias': 'mmlu', 'repo': 'cais/mmlu', 'config': 'all', 'revision': 'main', 'default_split': 'test', 'adapter': 'mmlu'}


## Laden und filtern

`limit` zählt akzeptierte Fragen nach der Filterung.


In [3]:
from src.data.registry import load_samples

config = named_config("datasets", "medqa")
samples, fingerprint = load_samples(config, "test", limit=5)

print(f"{len(samples)} Samples geladen | dataset fingerprint = {fingerprint}")
print("Typ:", type(samples[0]).__name__)


5 Samples geladen | dataset fingerprint = 9e4e5c3c66e6bc35
Typ: MCQSample


## Datenschema

`sample_id`, `dataset`, `subject`, `question`, `options[4]`, `gold_index` (0–3). `gold_label` wird daraus abgeleitet.


In [ ]:
# Ein Beispiel pro Datensatz: medqa, medmcqa, mmlu.
# Jeder Datensatz wird mit seinem eigenen Standard-Split geladen
# (medmcqa hat auf 'test' keine Labels -> default_split = 'validation').
from src.data.registry import load_samples

def lade_erstes(alias):
    cfg = named_config("datasets", alias)
    ds_samples, _ = load_samples(cfg, cfg["default_split"], limit=1)
    return ds_samples[0]

s_medqa   = lade_erstes("medqa")     # ein Sample aus MedQA
s_medmcqa = lade_erstes("medmcqa")   # ein Sample aus MedMCQA
s_mmlu    = lade_erstes("mmlu")      # ein Sample aus MMLU

def zeige_sample(s):
    print("sample_id :", s.sample_id)
    print("dataset   :", s.dataset)
    print("subject   :", s.subject)
    print("gold_index:", s.gold_index, " -> gold_label:", s.gold_label)
    print("\nFrage:\n", s.question[:300], "...")
    print("\nOptionen:")
    for i, opt in enumerate(s.options):
        star = "  <-- richtig" if i == s.gold_index else ""
        print(f"  [{i}] {'ABCD'[i]}) {opt}{star}")

for s in (s_medqa, s_medmcqa, s_mmlu):
    print("=" * 72)
    print(f"### {s.dataset.upper()}")
    print("=" * 72)
    zeige_sample(s)
    print()

# fuer die folgende Zelle (to_dict-Beispiel) behalten wir ein Sample in `s`
s = s_medqa

In [5]:
# to_dict() ist die serialisierbare Form (spaeter in Parquet gespeichert)
import json
print(json.dumps(s.to_dict(), indent=2, ensure_ascii=False)[:700], "...")


{
  "sample_id": "test-00000",
  "dataset": "medqa",
  "question": "A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?",
  "options": [
    "D ...


Die Antwortidentität bleibt der ursprüngliche Inhaltsindex; A–D ändern sich mit der Permutation. Doppelte Optionstexte werden ausgeschlossen.
